# Member 2: train M0 / M1 / M2 (FloodVerify CV)
Run top to bottom. Every training cell writes `last.pt` to Drive after each epoch; if Colab
disconnects, re-run setup + data, then re-run the same training cell and it resumes.
Paste real outputs to the team; never quote numbers you did not see here.

In [ ]:
# Runtime > Change runtime type > T4 GPU first.
BRANCH = "m3/eval-export"   # top of the CV branch stack; switch to "main" once the PRs are merged
from google.colab import drive
drive.mount("/content/drive")
!rm -rf /content/FloodVerify
!git clone -q -b {BRANCH} https://github.com/aaarrvind/FloodVerify.git /content/FloodVerify
%cd /content/FloodVerify
!pip install -q -e . --no-deps jsonschema
import sys; sys.path.insert(0, "/content/FloodVerify/src")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Copy xBD to local disk (10-15 min). Skip if /content/xbd already exists in this session.
D = "/content/drive/MyDrive/MAAI 902 X 901 PROJECT"
CK = f"{D}/checkpoints"          # run folders go here; never in git
X, MAN = "/content/xbd", "/content/xbd/rscc_flood_manifest.csv"
import os
if not os.path.exists(f"{X}/processed/evidence_labels"):
    !mkdir -p /content/xbd/processed
    !tar -xf "{D}/xbd_tars/train_images_labels_targets.tar" -C /content/xbd
    !tar -xf "{D}/xbd_tars/test_images_labels_targets.tar" -C /content/xbd
    !cp -r "{D}/xbd/hold" /content/xbd/
    !cp -r "{D}/xbd/tier3" /content/xbd/
    !cp "{D}/xbd/rscc_flood_manifest.csv" /content/xbd/
    !unzip -q "{D}/xbd/processed.zip" -d /content/xbd/processed
!ls /content/xbd/processed/evidence_labels | wc -l   # expect 10148

## Task 1: loader check (expect train 5488, val 2184, test 2476)

In [ ]:
from floodsitrep.data.dataset import XbdTiles
from torch.utils.data import DataLoader
for role in ("train", "val", "test"):
    print(role, len(XbdTiles(X, MAN, role=role)))
b = next(iter(DataLoader(XbdTiles(X, MAN, role="train", augment=True), batch_size=8, shuffle=True, num_workers=2)))
print(b["pre"].shape, b["post"].shape, b["mask"].shape, b["mask"].unique().tolist(), b["valid"].float().mean().item())

## Task 2: forward pass returns a 5-class map the size of the mask

In [ ]:
import torch
from floodsitrep.cv.models import build_model
for name in ("early_fusion_unet", "siamese_unet"):
    m = build_model({"model": name}).cuda().eval()
    with torch.no_grad(), torch.autocast("cuda"):
        out = m(b["pre"][:2].cuda(), b["post"][:2].cuda())
    print(name, tuple(out.shape), sum(p.numel() for p in m.parameters()) / 1e6, "M params")
del m; torch.cuda.empty_cache()

## Task 3 / Gate 1: smoke run (320 train tiles, 96 val tiles, 2 epochs)
Done when: loss falls across the step logs, and `last.pt`, `best.pt`, `metrics.jsonl` exist.
Note the seconds per epoch: full-run time is roughly that x (5488 / 320).

In [ ]:
!python -m floodsitrep.cv.train --config configs/cv/smoke_m1.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}"
!ls -la "{CK}/smoke_m1_s0"

## Task 4: full M1 run (Dice + focal), all train tiles, validation on Florence
Re-run this cell after a disconnect to resume. Lower `epochs` with `--set epochs=12` if time is short
(write the change down: it is part of the run record).

In [ ]:
!python -m floodsitrep.cv.train --config configs/cv/m1_siamese_unet.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}"

## Task 5: loss comparison partner (weighted cross-entropy, same seed)

In [ ]:
!python -m floodsitrep.cv.train --config configs/cv/m1_siamese_unet_ce.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}"

## M0 early fusion, then M2 SegFormer (cut M2 first if late). Extra seeds: `--set seed=1` (run name gets `_s1`).

In [ ]:
!python -m floodsitrep.cv.train --config configs/cv/m0_early_fusion_unet.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}"

In [ ]:
!python -m floodsitrep.cv.train --config configs/cv/m2_siamese_segformer.yaml --xbd-root {X} --manifest {MAN} --out-dir "{CK}"

## Read a run's log

In [ ]:
import json, pandas as pd
RUN = "m1_siamese_unet_s0"
rows = [json.loads(l) for l in open(f"{CK}/{RUN}/metrics.jsonl")]
pd.DataFrame(rows)[["epoch", "train_loss", "val_loss", "xview2_pixel_score", "loc_f1", "damage_f1_hm",
                    "dmg_f1_minor", "dmg_f1_major", "dmg_f1_destroyed", "seconds"]].round(4)